# DataFrame Fundamentals

## Module 3, Notebook 1

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Explain** what a DataFrame is and how it relates to RDDs and Datasets
2. **Create** DataFrames from multiple sources: RDDs, Python collections, files, and explicit schemas
3. **Perform** basic DataFrame operations: `select`, `filter`, `withColumn`, `drop`, `alias`
4. **Work with** Column objects and column expressions
5. **Inspect** DataFrames with `show()`, `printSchema()`, `dtypes`, `columns`
6. **Compare** Spark DataFrames with pandas DataFrames

## Why DataFrames?

While RDDs gave us **fault-tolerant distributed computation**, they had a critical limitation:
Spark couldn't optimize your code because it didn't know **what** your data looked like or **what** you were doing with it.

DataFrames solve this by adding **structure** — named columns with types — enabling Spark's Catalyst optimizer
to rewrite and optimize your queries automatically, just like a relational database.

```
Evolution of Spark APIs:

  RDD (2011)          DataFrame (2015)        Dataset (2016)
  ┌──────────┐        ┌──────────────┐        ┌──────────────┐
  │ Untyped  │        │ Typed Columns│        │ Typed Objects │
  │ Objects  │───────►│ + Catalyst   │───────►│ + Compile-time│
  │ No optim.│        │ + Tungsten   │        │   type safety │
  └──────────┘        └──────────────┘        └──────────────┘
       ▲                     ▲                       ▲
  Low-level API        High-level API          Scala/Java only
  Full control         Auto-optimized          Python uses DF
```

## Architecture: DataFrame Under the Hood

A DataFrame is fundamentally a **distributed collection of rows organized into named columns**.
It is equivalent to a table in a relational database or a DataFrame in pandas — but distributed across a cluster.

```
┌─────────────────────────────────────────────────────────┐
│                    DataFrame API                         │
│   df.select("name","age").filter(col("age") > 30)      │
└───────────────────────┬─────────────────────────────────┘
                        │
                        ▼
┌─────────────────────────────────────────────────────────┐
│              Catalyst Optimizer                          │
│  1. Parse → Unresolved Logical Plan                     │
│  2. Analyze → Resolved Logical Plan                     │
│  3. Optimize → Optimized Logical Plan                   │
│  4. Plan → Physical Plans → Best Physical Plan          │
└───────────────────────┬─────────────────────────────────┘
                        │
                        ▼
┌─────────────────────────────────────────────────────────┐
│              Tungsten Execution Engine                    │
│  • Off-heap memory management                           │
│  • Cache-aware computation                              │
│  • Whole-stage code generation (codegen)                │
└───────────────────────┬─────────────────────────────────┘
                        │
                        ▼
┌─────────────────────────────────────────────────────────┐
│              RDD (Internal Execution)                    │
│  Partition 0  │  Partition 1  │  Partition 2  │  ...    │
│  [Row, Row..] │  [Row, Row..] │  [Row, Row..] │         │
└─────────────────────────────────────────────────────────┘
```

### Key Insight

Every DataFrame operation builds up a **logical plan** (a tree of transformations).
When you trigger an action, Catalyst optimizes that plan before executing it on the cluster.
This means **two different ways of writing the same query can produce the same physical plan**.

## DataFrame vs RDD vs Dataset

| Feature | RDD | DataFrame | Dataset |
|---------|-----|-----------|----------|
| **Type Safety** | Compile-time (Scala) | Runtime only | Compile-time (Scala/Java) |
| **Optimization** | None — manual | Catalyst + Tungsten | Catalyst + Tungsten |
| **Schema** | No schema | Yes — named, typed columns | Yes — encoded types |
| **Serialization** | Java/Kryo | Tungsten binary | Tungsten binary |
| **Language** | Python, Scala, Java, R | Python, Scala, Java, R | Scala, Java only |
| **API Style** | Functional (map, filter) | Declarative (select, where) | Both |
| **Performance** | Slowest (Python UDFs) | Fast (codegen) | Fast (codegen) |

> **In PySpark, DataFrame IS the Dataset API.** Python doesn't have compile-time types,
> so `Dataset[Row]` in Scala is simply `DataFrame` in Python. You get all the optimization
> benefits without needing the Dataset API explicitly.

---
## Setup: Create SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("03.01 - DataFrame Fundamentals")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"Application ID: {spark.sparkContext.applicationId}")

---
## 1⃣ Creating DataFrames

There are multiple ways to create DataFrames. Let's explore each one.

### 1.1 From a Python List (with `createDataFrame`)

The simplest way — great for testing and prototyping.

In [ ]:
# Method 1: List of tuples + column names
data = [
    ("Alice", 34, "Engineering"),
    ("Bob", 28, "Marketing"),
    ("Carol", 45, "Engineering"),
    ("Dave", 31, "Sales"),
    ("Eve", 29, "Marketing")
]

df_from_list = spark.createDataFrame(data, schema=["name", "age", "department"])
df_from_list.show()

In [ ]:
# Method 2: List of Row objects
from pyspark.sql import Row

rows = [
    Row(name="Alice", age=34, department="Engineering"),
    Row(name="Bob", age=28, department="Marketing"),
    Row(name="Carol", age=45, department="Engineering")
]

df_from_rows = spark.createDataFrame(rows)
df_from_rows.show()

# Row objects preserve field names automatically
print("Schema:")
df_from_rows.printSchema()

In [ ]:
# Method 3: List of dictionaries
dict_data = [
    {"name": "Alice", "salary": 95000.0, "active": True},
    {"name": "Bob", "salary": 72000.0, "active": False},
    {"name": "Carol", "salary": 110000.0, "active": True}
]

df_from_dicts = spark.createDataFrame(dict_data)
df_from_dicts.show()
df_from_dicts.printSchema()

### 1.2 From an RDD

You can convert any RDD to a DataFrame by adding schema information.

In [ ]:
# Create an RDD first
rdd = spark.sparkContext.parallelize([
    ("Laptop", 999.99, 150),
    ("Mouse", 29.99, 500),
    ("Keyboard", 79.99, 300),
    ("Monitor", 449.99, 100)
])

# Convert RDD to DataFrame with column names
df_from_rdd = rdd.toDF(["product", "price", "stock"])
df_from_rdd.show()

# Note: Spark infers types from the Python values
df_from_rdd.printSchema()

In [ ]:
# Convert RDD of Row objects (more explicit)
rdd_rows = spark.sparkContext.parallelize([
    Row(product="Laptop", price=999.99, stock=150),
    Row(product="Mouse", price=29.99, stock=500)
])

df_from_rdd_rows = spark.createDataFrame(rdd_rows)
df_from_rdd_rows.show()

### 1.3 From an Explicit Schema

In production, you should **always** define schemas explicitly. This avoids a scan of the data for type inference.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Define schema explicitly
product_schema = StructType([
    StructField("product", StringType(), nullable=False),
    StructField("price", DoubleType(), nullable=False),
    StructField("stock", IntegerType(), nullable=True)
])

data = [
    ("Laptop", 999.99, 150),
    ("Mouse", 29.99, 500),
    ("Keyboard", 79.99, None)   # stock is nullable
]

df_with_schema = spark.createDataFrame(data, schema=product_schema)
df_with_schema.show()
df_with_schema.printSchema()

In [ ]:
# You can also use DDL-style strings for schemas (quick and readable)
ddl_schema = "product STRING, price DOUBLE, stock INT"

df_ddl = spark.createDataFrame(data, schema=ddl_schema)
df_ddl.printSchema()

### 1.4 From Files (CSV, JSON, Parquet, etc.)

The most common way to create DataFrames in real-world applications.

In [ ]:
# Read a CSV file — the retail orders dataset
orders = (
    spark.read
    .option("header", "true")      # First row is header
    .option("inferSchema", "true")  # Infer column types (scans data)
    .csv("/opt/spark-data/retail/orders.csv")
)

print(f"Number of rows: {orders.count()}")
print(f"Number of columns: {len(orders.columns)}")
orders.show(5)

In [ ]:
# Read with explicit schema (recommended for production)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType
)

orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("order_date", StringType(), True),
    StructField("status", StringType(), True)
])

orders_explicit = (
    spark.read
    .option("header", "true")
    .schema(orders_schema)
    .csv("/opt/spark-data/retail/orders.csv")
)

orders_explicit.printSchema()
orders_explicit.show(5)

In [ ]:
# Read JSON file
events = spark.read.json("/opt/spark-data/retail/order_events.json")
events.printSchema()
events.show(3, truncate=False)

### 1.5 Creating Empty DataFrames and Single-Row DataFrames

In [ ]:
# Empty DataFrame with schema (useful for initializing accumulation patterns)
empty_schema = StructType([
    StructField("id", IntegerType()),
    StructField("value", StringType())
])

empty_df = spark.createDataFrame([], schema=empty_schema)
print(f"Empty DataFrame count: {empty_df.count()}")
empty_df.printSchema()

In [ ]:
# range() — creates a DataFrame with a single 'id' column
range_df = spark.range(0, 10, 2)  # start=0, end=10, step=2
range_df.show()
print(f"Partitions: {range_df.rdd.getNumPartitions()}")

---
## 2⃣ Inspecting DataFrames

Before you do anything with a DataFrame, you need to understand its structure and content.

In [ ]:
# Let's work with the orders dataset from here on
orders = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/opt/spark-data/retail/orders.csv")
)

# 1. show() — displays rows as a formatted table
orders.show(5)                 # Default: 20 rows, truncate=True
orders.show(5, truncate=False) # Don't truncate long values
orders.show(5, truncate=30)    # Truncate at 30 chars

In [ ]:
# 2. printSchema() — shows the schema tree
orders.printSchema()

In [ ]:
# 3. schema — returns StructType object (programmatic access)
schema = orders.schema
print(f"Schema type: {type(schema)}")
print()

# Iterate over fields
for field in schema.fields:
    print(f"  {field.name:15s} | {str(field.dataType):15s} | nullable={field.nullable}")

In [ ]:
# 4. columns — list of column names
print("Columns:", orders.columns)
print(f"Number of columns: {len(orders.columns)}")

In [ ]:
# 5. dtypes — list of (column_name, type_string) tuples
print("Column types:")
for col_name, col_type in orders.dtypes:
    print(f"  {col_name}: {col_type}")

In [ ]:
# 6. describe() — summary statistics for numeric columns
orders.describe().show()

In [ ]:
# 7. summary() — more detailed statistics including percentiles
orders.summary("count", "min", "max", "mean", "25%", "50%", "75%").show()

In [ ]:
# 8. count() — number of rows
print(f"Row count: {orders.count()}")

In [ ]:
# 9. first() and head() — get the first row(s)
first_row = orders.first()
print(f"First row: {first_row}")
print(f"Order ID: {first_row['order_id']}")

# head(n) returns a list of Row objects
top3 = orders.head(3)
for row in top3:
    print(f"  Order {row['order_id']}: status={row['status']}")

In [ ]:
# 10. take() — similar to head() but always returns a list
rows = orders.take(2)
print(f"Took {len(rows)} rows")
print(f"Type: {type(rows[0])}")

---
## 3⃣ Column Objects and Expressions

Understanding **Column objects** is fundamental to working with DataFrames.
A Column represents a column expression — it's not actual data, but a **recipe** for computing values.

```
Column Object — What it represents:

  col("price") * col("quantity")
       │              │
       ▼              ▼
  ┌─────────┐   ┌──────────┐
  │ Column  │   │  Column  │
  │"price"│ * │"quantity"│  ──► New Column expression
  └─────────┘   └──────────┘
  
  This doesn't compute anything yet!
  It creates a plan that Spark will execute later.
```

In [ ]:
from pyspark.sql.functions import col, column, expr

# There are FOUR ways to reference a column:

# Way 1: col() function (most common, recommended)
c1 = col("price")
print(f"col():    {c1}, type: {type(c1)}")

# Way 2: column() function (same as col, just longer)
c2 = column("price")
print(f"column(): {c2}, type: {type(c2)}")

# Way 3: df["column_name"] — tied to a specific DataFrame
c3 = orders["price"]
print(f"df[]:     {c3}, type: {type(c3)}")

# Way 4: df.column_name — convenient but limited (no special chars)
c4 = orders.price
print(f"df.col:   {c4}, type: {type(c4)}")

In [ ]:
# Column expressions — building computation recipes
from pyspark.sql.functions import col, lit

# Arithmetic on columns
total_expr = col("price") * col("quantity")
print(f"Expression: {total_expr}")

# Comparison (creates a boolean Column)
expensive = col("price") > 100
print(f"Comparison: {expensive}")

# lit() creates a Column with a constant value
tax_rate = lit(0.08)
print(f"Literal: {tax_rate}")

# Complex expression
total_with_tax = (col("price") * col("quantity")) * (lit(1) + lit(0.08))
print(f"Complex: {total_with_tax}")

In [ ]:
# expr() — write column expressions as SQL strings
from pyspark.sql.functions import expr

# These two are equivalent:
expr1 = col("price") * col("quantity")
expr2 = expr("price * quantity")

# expr() is useful for complex SQL expressions
expr3 = expr("CASE WHEN price > 100 THEN 'expensive' ELSE 'affordable' END")

# Let's use them
orders.select(
    col("order_id"),
    col("price"),
    col("quantity"),
    expr1.alias("total_v1"),
    expr2.alias("total_v2"),
    expr3.alias("price_category")
).show(5)

---
## 4⃣ Core DataFrame Operations

These are the bread-and-butter operations you'll use constantly.

```
DataFrame Operations Cheat Sheet:

┌──────────────────────────────────────────────────────┐
│  PROJECTION (columns)    │  SELECTION (rows)         │
│  ────────────────────    │  ─────────────────        │
│  select()                │  filter() / where()       │
│  selectExpr()            │  limit()                  │
│  withColumn()            │  distinct()               │
│  withColumnRenamed()     │  dropDuplicates()         │
│  drop()                  │  sample()                 │
│  alias()                 │                           │
├──────────────────────────┼───────────────────────────┤
│  ORDERING                │  COMBINING                │
│  ────────                │  ─────────                │
│  orderBy() / sort()      │  union() / unionAll()     │
│  sortWithinPartitions()  │  unionByName()            │
│                          │  join()                   │
│                          │  crossJoin()              │
└──────────────────────────┴───────────────────────────┘
```

### 4.1 select() — Choosing Columns

In [ ]:
# select() picks specific columns (like SQL SELECT)

# Using column name strings
orders.select("order_id", "price", "quantity").show(5)

# Using Column objects
orders.select(col("order_id"), col("price"), col("quantity")).show(5)

# Mixing strings and Column expressions
orders.select(
    "order_id",
    col("price"),
    (col("price") * col("quantity")).alias("total")
).show(5)

In [ ]:
# selectExpr() — select with SQL expression strings
orders.selectExpr(
    "order_id",
    "price * quantity AS total",
    "UPPER(status) AS status_upper",
    "CASE WHEN price > 100 THEN 'High' ELSE 'Low' END AS tier"
).show(5)

### 4.2 filter() / where() — Selecting Rows

In [ ]:
# filter() and where() are identical — use whichever reads better

# Using Column expressions
expensive_orders = orders.filter(col("price") > 100)
print(f"Orders with price > 100: {expensive_orders.count()}")

# Using SQL expression string
expensive_orders2 = orders.where("price > 100")
print(f"Same result: {expensive_orders2.count()}")

In [ ]:
# Multiple conditions with & (AND), | (OR), ~ (NOT)
# IMPORTANT: Each condition must be in parentheses!

filtered = orders.filter(
    (col("price") > 50) &
    (col("quantity") >= 2) &
    (col("status") == "shipped")
)
print(f"Filtered count: {filtered.count()}")
filtered.show(5)

In [ ]:
# Useful filter methods

# isin() — check membership
orders.filter(col("status").isin("shipped", "delivered")).show(5)

# contains(), startswith(), endswith()
orders.filter(col("status").startswith("ship")).show(5)

# between()
orders.filter(col("price").between(50, 200)).show(5)

# isNull() / isNotNull()
orders.filter(col("price").isNotNull()).count()

### 4.3 withColumn() — Adding or Modifying Columns

In [ ]:
from pyspark.sql.functions import col, lit, when, round as spark_round

# Add a new column
orders_enhanced = (
    orders
    .withColumn("total", col("price") * col("quantity"))
    .withColumn("tax", spark_round(col("price") * col("quantity") * 0.08, 2))
    .withColumn("currency", lit("USD"))
)

orders_enhanced.select("order_id", "price", "quantity", "total", "tax", "currency").show(5)

In [ ]:
# Modify an existing column (replaces in-place by name)
orders_upper = orders.withColumn("status", col("status"))

# Conditional column with when/otherwise
orders_tiered = orders.withColumn(
    "price_tier",
    when(col("price") < 50, "Budget")
    .when(col("price") < 200, "Mid-range")
    .when(col("price") < 500, "Premium")
    .otherwise("Luxury")
)

orders_tiered.select("order_id", "price", "price_tier").show(10)

### 4.4 withColumnRenamed() and alias()

In [ ]:
# Rename columns
orders_renamed = (
    orders
    .withColumnRenamed("order_id", "id")
    .withColumnRenamed("customer_id", "cust_id")
)
orders_renamed.columns

In [ ]:
# alias() — rename a column expression in select
orders.select(
    col("order_id").alias("id"),
    col("price").alias("unit_price"),
    (col("price") * col("quantity")).alias("line_total")
).show(5)

In [ ]:
# DataFrame alias() — useful for self-joins and disambiguation
o = orders.alias("o")
o.select("o.order_id", "o.price").show(3)

### 4.5 drop() — Removing Columns

In [ ]:
# Drop one or more columns
orders_slim = orders.drop("customer_id", "product_id")
print(f"Original columns: {orders.columns}")
print(f"After drop:       {orders_slim.columns}")

In [ ]:
# Drop using Column object
orders.drop(col("status")).columns

### 4.6 distinct() and dropDuplicates()

In [ ]:
# distinct() — unique rows (considers ALL columns)
unique_statuses = orders.select("status").distinct()
unique_statuses.show()
print(f"Unique statuses: {unique_statuses.count()}")

In [ ]:
# dropDuplicates() — keep first occurrence, can specify subset of columns
# Get one order per customer (first occurrence)
one_per_customer = orders.dropDuplicates(["customer_id"])
print(f"Original rows: {orders.count()}")
print(f"After dedup on customer_id: {one_per_customer.count()}")

### 4.7 orderBy() / sort()

In [ ]:
from pyspark.sql.functions import desc, asc

# Sort ascending (default)
orders.orderBy("price").show(5)

# Sort descending
orders.orderBy(col("price").desc()).show(5)

# Sort by multiple columns
orders.orderBy(desc("price"), asc("order_id")).show(5)

In [ ]:
# Null ordering
orders.orderBy(col("price").asc_nulls_first()).show(5)   # NULLs first
orders.orderBy(col("price").desc_nulls_last()).show(5)    # NULLs last

### 4.8 limit() and sample()

In [ ]:
# limit() — get first N rows as a DataFrame
top10 = orders.orderBy(desc("price")).limit(10)
top10.show()

# sample() — random sample
sampled = orders.sample(fraction=0.01, seed=42)
print(f"Sample size: {sampled.count()} (from {orders.count()})")

---
## 5⃣ Chaining Transformations

DataFrame operations return new DataFrames, enabling a **fluent** or **chaining** style.
This is one of the most powerful patterns in Spark.

In [ ]:
from pyspark.sql.functions import col, when, round as spark_round, desc

# A complete transformation pipeline
result = (
    orders
    # Filter to shipped orders
    .filter(col("status") == "shipped")
    # Add computed columns
    .withColumn("total", spark_round(col("price") * col("quantity"), 2))
    .withColumn("tier",
        when(col("price") < 50, "Budget")
        .when(col("price") < 200, "Mid")
        .otherwise("Premium")
    )
    # Select relevant columns
    .select("order_id", "customer_id", "total", "tier")
    # Sort by total descending
    .orderBy(desc("total"))
    # Top 10
    .limit(10)
)

result.show()

### Why Chaining Works

```
orders                      ← DataFrame
  .filter(...)              ← Returns new DataFrame
  .withColumn(...)          ← Returns new DataFrame
  .select(...)              ← Returns new DataFrame
  .orderBy(...)             ← Returns new DataFrame
  .limit(10)                ← Returns new DataFrame

None of these modify the original'orders'DataFrame!
DataFrames are IMMUTABLE — each operation returns a new one.
```

---
## 6⃣ Internal Investigation

Let's look at what Spark does behind the scenes.

In [ ]:
# explain() — see the execution plan
result = (
    orders
    .filter(col("price") > 100)
    .select("order_id", "price")
    .orderBy(desc("price"))
)

print("=== Simple explain ===")
result.explain()

print("\n=== Extended explain ===")
result.explain(mode="extended")

In [ ]:
# Formatted explain — the most readable
result.explain(mode="formatted")

In [ ]:
# Check partitioning
print(f"Orders partitions: {orders.rdd.getNumPartitions()}")

# After a shuffle operation (orderBy)
sorted_orders = orders.orderBy("price")
# Note: We need to trigger an action to see the actual partitions
print(f"After sort partitions: {sorted_orders.rdd.getNumPartitions()}")

In [ ]:
# glom() — see what data is in each partition
small_df = spark.createDataFrame(
    [(i, f"val_{i}") for i in range(12)],
    schema=["id", "value"]
).repartition(3)

partitions = small_df.rdd.glom().collect()
for i, part in enumerate(partitions):
    print(f"Partition {i} ({len(part)} rows): {[row['id'] for row in part]}")

In [ ]:
# Demonstrate that Catalyst optimizes different queries to the same plan

# Query 1: filter then select
q1 = orders.filter(col("price") > 100).select("order_id", "price")

# Query 2: select then filter
q2 = orders.select("order_id", "price").filter(col("price") > 100)

print("=== Query 1 plan ===")
q1.explain()

print("\n=== Query 2 plan ===")
q2.explain()

# Both produce the same physical plan! Catalyst is smart.

---
## 7⃣ Experiments

### Experiment 1: Schema Inference vs Explicit Schema Performance

In [ ]:
import time

# Test 1: With inferSchema (requires extra scan)
start = time.time()
df_infer = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/opt/spark-data/retail/orders.csv")
)
_ = df_infer.count()  # Force evaluation
infer_time = time.time() - start

# Test 2: With explicit schema (no extra scan needed)
start = time.time()
df_explicit = (
    spark.read
    .option("header", "true")
    .schema(orders_schema)
    .csv("/opt/spark-data/retail/orders.csv")
)
_ = df_explicit.count()  # Force evaluation
explicit_time = time.time() - start

print(f"With inferSchema:   {infer_time:.3f}s")
print(f"With explicit schema: {explicit_time:.3f}s")
print(f"Speedup: {infer_time/explicit_time:.2f}x")

### Experiment 2: Column Reference Methods — Are They Equivalent?

In [ ]:
# Hypothesis: All column reference methods produce the same plan

q_col = orders.select(col("order_id"), col("price"))
q_str = orders.select("order_id", "price")
q_bracket = orders.select(orders["order_id"], orders["price"])

# Compare plans
print("=== col() ===")
q_col.explain()
print("=== string ===")
q_str.explain()
print("=== bracket ===")
q_bracket.explain()

# Verify same results
assert q_col.collect() == q_str.collect() == q_bracket.collect()
print("\n All methods produce identical results!")

### Experiment 3: Immutability Verification

In [ ]:
# Hypothesis: Operations on a DataFrame don't modify the original

original = orders.select("order_id", "price", "status")
original_columns = original.columns.copy()
original_count = original.count()

# Perform operations
modified = original.withColumn("new_col", lit(1)).filter(col("price") > 100)

# Verify original is unchanged
assert original.columns == original_columns, "Columns changed!"
assert original.count() == original_count, "Count changed!"
assert "new_col" not in original.columns, "New column appeared in original!"

print(f"Original columns: {original.columns}")
print(f"Modified columns: {modified.columns}")
print(f"Original count: {original_count}, Modified count: {modified.count()}")
print("\n Original DataFrame is unchanged — immutability confirmed!")

---
## 8⃣ Spark DataFrames vs pandas DataFrames

| Feature | Spark DataFrame | pandas DataFrame |
|---------|----------------|------------------|
| **Scale** | TB+ across cluster | Single machine RAM |
| **Evaluation** | Lazy | Eager (immediate) |
| **Mutability** | Immutable | Mutable |
| **Index** | No row index | Has row index |
| **Operations** | Transformations + Actions | Direct operations |
| **Null handling** | null (any column) | NaN (numeric) / None |
| **Schema** | Strict schema | Flexible typing |
| **Optimization** | Catalyst auto-optimizes | Manual optimization |

In [ ]:
# Converting between Spark and pandas DataFrames

# Spark → pandas (CAUTION: collects ALL data to driver!)
small_spark_df = orders.limit(100)
pandas_df = small_spark_df.toPandas()
print(f"Type: {type(pandas_df)}")
print(f"Shape: {pandas_df.shape}")
print(pandas_df.head())

In [ ]:
# pandas → Spark
import pandas as pd

pdf = pd.DataFrame({
    "name": ["Alice", "Bob", "Carol"],
    "score": [95, 87, 92]
})

spark_df = spark.createDataFrame(pdf)
spark_df.show()
spark_df.printSchema()

### Common Pitfalls When Coming from pandas

1. **No row index** — Spark DataFrames don't have a row index. Use `monotonically_increasing_id()` if you need one.
2. **Lazy evaluation** — Nothing happens until you call an action (`show()`, `count()`, `collect()`).
3. **Immutability** — `df.withColumn(...)` returns a NEW DataFrame; it doesn't modify `df`.
4. **No iloc/loc** — You can't access rows by position. Use `filter()` instead.
5. **toPandas() is dangerous** — It collects ALL data to the driver. Only use on small DataFrames.

In [ ]:
# Common pandas habit that works differently in Spark

# pandas:  df['new_col'] = df['price'] * 2      ← Mutates in place
# Spark:   df = df.withColumn('new_col', col('price') * 2)  ← Returns new DF

# pandas:  df[df['price'] > 100]                 ← Boolean indexing
# Spark:   df.filter(col('price') > 100)         ← filter/where

# pandas:  df.iloc[5]                            ← Row by position
# Spark:   (Not directly supported — need workaround)

# pandas:  len(df)                               ← Instant
# Spark:   df.count()                            ← Triggers a job!

print("See the differences above — they're critical for Spark newcomers!")

---
## 9⃣ DataFrame Actions vs Transformations

This distinction is crucial for understanding Spark's execution model.

```
TRANSFORMATIONS (Lazy)          ACTIONS (Trigger Execution)
─────────────────────          ────────────────────────────
select()                       show()
filter() / where()             count()
withColumn()                   collect()
groupBy()                      first() / head() / take()
orderBy()                      toPandas()
join()                         write.csv() / write.parquet()
union()                        foreach()
distinct()                     describe()
```

In [ ]:
import time

# Transformations are INSTANT — they just build up the plan
start = time.time()
lazy_result = (
    orders
    .filter(col("price") > 100)
    .withColumn("total", col("price") * col("quantity"))
    .select("order_id", "total")
    .orderBy(desc("total"))
)
transform_time = time.time() - start

# Actions trigger the actual computation
start = time.time()
_ = lazy_result.count()  # THIS is when Spark actually does the work
action_time = time.time() - start

print(f"Transformation time: {transform_time:.6f}s (just planning)")
print(f"Action time:         {action_time:.3f}s (actual computation)")

---
## Exercises

Use the `/opt/spark-data/retail/orders.csv` dataset for all exercises.

### Exercise 1 (Beginner): Basic Inspection

1. Read the orders CSV file with `inferSchema=True`
2. Print the schema
3. Show the first 10 rows
4. Print the number of rows and columns
5. Get the list of column names
6. Run `describe()` on the DataFrame

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Beginner): Column Selection and Filtering

1. Select only `order_id`, `price`, and `status` columns
2. Filter orders where `status` is"delivered"
3. Find orders where `price` is between 100 and 500
4. Count orders where `quantity` is greater than 3

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Intermediate): Computed Columns

1. Add a `total` column = `price * quantity`
2. Add a `discount` column that gives 10% off for orders with quantity >= 5, 0% otherwise
3. Add a `final_total` = `total - (total * discount)`
4. Show the top 15 orders by `final_total` descending
5. Drop the intermediate `discount` column from the final result

In [ ]:
# YOUR CODE HERE


### Exercise 4 (Intermediate): Multiple Data Sources

1. Read `/opt/spark-data/retail/customers.csv` with header and inferSchema
2. Read `/opt/spark-data/retail/products.csv` with header and inferSchema
3. Print the schema of each
4. Count the rows in each
5. Show 5 sample rows from each
6. Find how many distinct `customer_id` values are in orders vs. customers

In [ ]:
# YOUR CODE HERE


### Exercise 5 (Advanced): Complete Pipeline

Build a single chained transformation pipeline that:
1. Reads orders.csv with an explicit schema (define it yourself)
2. Filters to only"shipped"and"delivered"statuses
3. Adds a `total` column (price × quantity)
4. Adds a `category` column:"small"(total < 100),"medium"(100-500),"large"(> 500)
5. Removes `product_id` column
6. Renames `customer_id` to `cust_id`
7. Sorts by total descending
8. Shows top 20 results

Then call `explain(mode='formatted')` on your result to see the plan.

In [ ]:
# YOUR CODE HERE


---
## Solutions

<details>
<summary>Click to expand solutions</summary>

### Solution 1: Basic Inspection
```python
orders = spark.read.option("header","true").option("inferSchema","true").csv("/opt/spark-data/retail/orders.csv")
orders.printSchema()
orders.show(10)
print(f"Rows: {orders.count()}, Columns: {len(orders.columns)}")
print(f"Column names: {orders.columns}")
orders.describe().show()
```

### Solution 2: Column Selection and Filtering
```python
from pyspark.sql.functions import col

orders.select("order_id","price","status").show(5)
orders.filter(col("status") =="delivered").show(5)
orders.filter(col("price").between(100, 500)).show(5)
print(orders.filter(col("quantity") > 3).count())
```

### Solution 3: Computed Columns
```python
from pyspark.sql.functions import col, when, desc, round as spark_round

result = (
    orders
    .withColumn("total", spark_round(col("price") * col("quantity"), 2))
    .withColumn("discount", when(col("quantity") >= 5, 0.10).otherwise(0.0))
    .withColumn("final_total", spark_round(col("total") - (col("total") * col("discount")), 2))
    .drop("discount")
    .orderBy(desc("final_total"))
    .limit(15)
)
result.show()
```

### Solution 4: Multiple Data Sources
```python
customers = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/customers.csv")
products = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/products.csv")

customers.printSchema()
products.printSchema()

print(f"Customers: {customers.count()}, Products: {products.count()}")
customers.show(5)
products.show(5)

print(f"Distinct customer_ids in orders: {orders.select('customer_id').distinct().count()}")
print(f"Distinct customer_ids in customers: {customers.select('customer_id').distinct().count()}")
```

### Solution 5: Complete Pipeline
```python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, when, desc, round as spark_round

schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("customer_id", IntegerType()),
    StructField("product_id", IntegerType()),
    StructField("quantity", IntegerType()),
    StructField("price", DoubleType()),
    StructField("order_date", StringType()),
    StructField("status", StringType())
])

result = (
    spark.read.option("header", True).schema(schema)
    .csv("/opt/spark-data/retail/orders.csv")
    .filter(col("status").isin("shipped","delivered"))
    .withColumn("total", spark_round(col("price") * col("quantity"), 2))
    .withColumn("category",
        when(col("total") < 100,"small")
        .when(col("total") <= 500,"medium")
        .otherwise("large")
)
    .drop("product_id")
    .withColumnRenamed("customer_id","cust_id")
    .orderBy(desc("total"))
)

result.show(20)
result.explain(mode="formatted")
```

</details>

---
## Summary

### Key Takeaways

| Concept | Details |
|---------|----------|
| **DataFrame** | Distributed collection of rows with named, typed columns |
| **Immutable** | Every operation returns a NEW DataFrame |
| **Lazy** | Transformations build plans; actions execute them |
| **Schema** | Always use explicit schemas in production |
| **Column** | An expression (recipe), not actual data |
| **Catalyst** | Optimizer that rewrites your queries automatically |

### Important Terms

- **DataFrame**: `Dataset[Row]` in Scala — a table-like distributed collection
- **Column**: An expression representing a computation on a column
- **Row**: A single record in a DataFrame
- **Schema**: `StructType` describing column names and types
- **Transformation**: Lazy operation that returns a new DataFrame
- **Action**: Eager operation that triggers computation

### Common Mistakes

1.  Using `inferSchema` in production (slow, can guess wrong)
2.  Forgetting parentheses in compound filters: `(cond1) & (cond2)`
3.  Calling `.toPandas()` on large DataFrames (OOM risk)
4.  Thinking `withColumn()` modifies the DataFrame in-place
5.  Using `==` for null comparison (use `.isNull()` instead)

### Interview Questions

1. What is the difference between a DataFrame and an RDD?
2. Why are DataFrames faster than RDDs for most operations?
3. What is Catalyst and how does it optimize DataFrame operations?
4. What is the difference between a transformation and an action?
5. Why should you use explicit schemas instead of `inferSchema`?

### Next Up

**Notebook 02: Schemas and Data Types** — Deep dive into Spark's type system, complex types, and schema enforcement.

In [ ]:
# Clean up
spark.stop()
print("SparkSession stopped.")